In [0]:
%pip install torch-geometric-signed-directed s3fs

In [0]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import f1_score
from torch_geometric.loader import DataLoader

# ============================================================
# Parameters (notebook widgets)
# ============================================================
dbutils.widgets.text("tag", "v1", "Experiment tag")
dbutils.widgets.text("n_users", "20", "Number of users")

EXPERIMENT_TAG = dbutils.widgets.get("tag")
N_USERS = int(dbutils.widgets.get("n_users"))

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}"
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"

# Storage mode: True = S3 user cache, False = local
USE_S3_STORAGE = True

S3_GNN_BUCKET = "pablocelayes-test"
S3_USER_CACHE_PREFIX = "learning/sna-classifier-gnn/user_samples_cache"

print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")
print(f"User cache: s3://{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}")

In [0]:
HISTORY_PATH = f"{EXPERIMENT_DIR}/training_history.pkl"

if not os.path.exists(HISTORY_PATH):
    raise FileNotFoundError(f"No history found at {HISTORY_PATH}")

with open(HISTORY_PATH, "rb") as f:
    history = pickle.load(f)

print(f"Loaded history: {len(history['step'])} checkpoints, {len(history['epoch_step'])} epochs")
print(f"  Last checkpoint step: {history['step'][-1] if history['step'] else 'N/A'}")
print(f"  Best val F1 (checkpoint): {max(history['val_f1']):.4f}" if history['val_f1'] else "")
print(f"  Best val F1 (end-of-epoch): {max(history['epoch_val_f1']):.4f}" if history['epoch_val_f1'] else "")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Loss curves ---
ax = axes[0]
if history["step"]:
    ax.plot(history["step"], history["train_loss"], 'b-', alpha=0.6, label='Train loss (checkpoint)')
    ax.plot(history["step"], history["val_loss"], 'r-', alpha=0.6, label='Val loss (checkpoint)')
if history["epoch_step"]:
    ax.plot(history["epoch_step"], history["epoch_val_loss"], 'ro-', markersize=5, label='Val loss (end-of-epoch)')
ax.set_xlabel('Global Step')
ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Right: F1 curves ---
ax = axes[1]
if history["step"]:
    ax.plot(history["step"], history["val_f1"], 'r-', alpha=0.6, label='Val F1 (checkpoint)')
if history["epoch_step"]:
    ax.plot(history["epoch_step"], history["epoch_train_f1"], 'b^-', markersize=5, label='Train F1 (end-of-epoch)')
    ax.plot(history["epoch_step"], history["epoch_val_f1"], 'ro-', markersize=5, label='Val F1 (end-of-epoch)')
ax.set_xlabel('Global Step')
ax.set_ylabel('F1 Score')
ax.set_title('Training & Validation F1')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'GNN Training Curves — {FINAL_TAG}', fontsize=13)
plt.tight_layout()
plt.show()

In [0]:
from gnn_models import PretrainedEmbeddingLookup, RetweetDataset, ParquetGNNLoader, RetweetGNN, evaluate

DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
# Load best model
BEST_MODEL_PATH = f"{EXPERIMENT_DIR}/best_retweet_gnn_general.pt"
assert os.path.exists(BEST_MODEL_PATH), f"No model found at {BEST_MODEL_PATH}"

model = RetweetGNN(
    ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH, device=device
).to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True, map_location=device))
model.eval()
print(f"Loaded model from {BEST_MODEL_PATH}")

# ---------------------------------------------------------------------------
# Load test samples directly from per-user cache (no experiment-level parquet)
# ---------------------------------------------------------------------------
import json
import s3fs
import pyarrow.parquet as pq
from random import shuffle as _shuffle_list

_loader_fs = s3fs.S3FileSystem() if USE_S3_STORAGE else None

# Load experiment manifest to know which users belong to this experiment
MANIFEST_PATH = f"{EXPERIMENT_DIR}/gnn_experiment_manifest.json"
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH) as f:
        experiment_manifest = json.load(f)
    _test_uids_from_train = [u["uid"] for u in experiment_manifest.get("train_users", [])]
    _test_uids_from_test = [u["uid"] for u in experiment_manifest.get("test_users", [])]
    print(f"Loaded manifest: {len(_test_uids_from_train)} train-group users, "
          f"{len(_test_uids_from_test)} test-group users")
else:
    # Fallback: use all users in cache
    print(f"No manifest at {MANIFEST_PATH}, using all users in cache")
    _user_cache_prefix = f"{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}"
    _user_dirs = _loader_fs.ls(_user_cache_prefix, detail=False)
    _test_uids_from_train = [d.split('/')[-1] for d in _user_dirs]
    _test_uids_from_test = []

# Build a simple streaming test loader from user cache
class _EvalUserCacheLoader:
    """Lightweight loader for evaluation: reads all test parquet from user cache."""
    def __init__(self, user_ids, splits, batch_size, fs):
        """
        user_ids: list of str UIDs
        splits: list of split names to read per user (e.g. ["test"] or ["train", "test"])
        """
        self.batch_size = batch_size
        self.fs = fs
        self.file_list = []  # (uid, path)
        self._total_samples = 0
        for uid in user_ids:
            for split in splits:
                path = f"{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}/{uid}/{split}.snappy.parquet"
                if fs is not None and fs.exists(path):
                    with fs.open(path, 'rb') as f:
                        n = pq.ParquetFile(f).metadata.num_rows
                    self.file_list.append((uid, path, n))
                    self._total_samples += n

    @property
    def total_samples(self):
        return self._total_samples

    def __len__(self):
        return self._total_samples // self.batch_size

    def __iter__(self):
        for uid, path, n in self.file_list:
            with self.fs.open(path, 'rb') as f:
                table = pq.read_table(f)
            for start in range(0, table.num_rows - self.batch_size + 1, self.batch_size):
                yield table.slice(start, self.batch_size)
            del table

# Test set = train-group users' "test" split + test-group users' both splits
_loaders = []
_loaders.append(_EvalUserCacheLoader(_test_uids_from_train, ["test"], batch_size=32, fs=_loader_fs))
if _test_uids_from_test:
    _loaders.append(_EvalUserCacheLoader(_test_uids_from_test, ["train", "test"], batch_size=32, fs=_loader_fs))

# Composite
class _CompositeLoader:
    def __init__(self, loaders):
        self.loaders = loaders
        self._total = sum(l.total_samples for l in loaders)
    @property
    def total_samples(self):
        return self._total
    def __len__(self):
        return self._total // 32
    def __iter__(self):
        for l in self.loaders:
            yield from l

test_loader = _CompositeLoader(_loaders) if len(_loaders) > 1 else _loaders[0]
print(f"Test loader (from user cache): {test_loader.total_samples} samples, {len(test_loader)} batches")

In [0]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Evaluate on global test set (streaming from parquet)
global_f1, global_preds, global_labels, _ = evaluate(model, test_loader, device)
print(f"=== GNN Global Test F1: {global_f1:.4f} ===")
print(f"  Total test samples: {test_loader.total_samples}")
print(f"  Positive rate: {global_labels.float().mean():.3f}")

In [0]:
# Compute F1 per user on their test samples (read directly from user cache)
gnn_f1s = {}

# All users that contribute to the test set
_eval_user_ids = _test_uids_from_train + _test_uids_from_test
print(f"Evaluating per-user F1 for {len(_eval_user_ids)} users from user cache...")

for uid_str in _eval_user_ids:
    # Read this user's test parquet from cache
    test_path = f"{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}/{uid_str}/test.snappy.parquet"
    if not _loader_fs.exists(test_path):
        continue
    with _loader_fs.open(test_path, 'rb') as f:
        table = pq.read_table(f)
    if table.num_rows == 0:
        continue

    samples = []
    for row_idx in range(table.num_rows):
        edge_src = table.column('edge_src')[row_idx].as_py()
        edge_dst = table.column('edge_dst')[row_idx].as_py()
        if edge_src:
            edge_index = np.column_stack([
                np.array(edge_src, dtype=np.int32),
                np.array(edge_dst, dtype=np.int32),
            ])
        else:
            edge_index = np.empty((0, 2), dtype=np.int32)
        samples.append({
            "central_user_id": int(table.column('central_user_id')[row_idx].as_py()),
            "neighbor_ids": np.array(table.column('neighbor_ids')[row_idx].as_py(), dtype=np.int64),
            "retweeted_ids": np.array(table.column('retweeted_ids')[row_idx].as_py(), dtype=np.int64),
            "edge_index": edge_index,
            "label": int(table.column('label')[row_idx].as_py()),
        })
    del table

    user_ds = RetweetDataset(samples)
    user_loader = DataLoader(user_ds, batch_size=32, shuffle=False)
    user_f1, _, _, _ = evaluate(model, user_loader, device)
    gnn_f1s[int(uid_str)] = user_f1

gnn_f1_values = list(gnn_f1s.values())

print(f"=== GNN — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(gnn_f1_values):.4f}")
print(f"  Median: {np.median(gnn_f1_values):.4f}")
print(f"  Std:    {np.std(gnn_f1_values):.4f}")
print(f"  Min:    {np.min(gnn_f1_values):.4f}")
print(f"  Max:    {np.max(gnn_f1_values):.4f}")
print(f"  Users:  {len(gnn_f1_values)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(gnn_f1_values, bins=20, edgecolor='black', alpha=0.7, color='darkorange')
ax.axvline(np.mean(gnn_f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(gnn_f1_values):.3f}')
ax.axvline(np.median(gnn_f1_values), color='blue', linestyle='--', label=f'Median: {np.median(gnn_f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title(f'GNN (General) — Per-user Test F1 Distribution — {FINAL_TAG}')
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# Load baseline SVC results from shared per-user cache (independent of GNN experiment)
BASELINE_USER_CACHE_PATH = f"{DATA_PATH}/baseline_svc_user_cache.pkl"

if not os.path.exists(BASELINE_USER_CACHE_PATH):
    print(f"No baseline user cache found at {BASELINE_USER_CACHE_PATH} — skipping comparison.")
else:
    with open(BASELINE_USER_CACHE_PATH, "rb") as f:
        baseline_user_cache = pickle.load(f)

    # Assemble experiment-level results from cache for users that overlap with GNN
    baseline_users = [uid for uid in gnn_f1s.keys() if uid in baseline_user_cache]
    baseline_f1s = {uid: baseline_user_cache[uid]["f1"] for uid in baseline_users}
    all_baseline_test_preds = [
        (baseline_user_cache[uid]["preds"], baseline_user_cache[uid]["labels"])
        for uid in baseline_users
    ]
    print(f"Loaded baseline cache: {len(baseline_user_cache)} users total, "
          f"{len(baseline_users)} overlap with GNN experiment")

    # Combined baseline F1
    all_bl_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
    all_bl_labels = np.concatenate([l for _, l in all_baseline_test_preds])
    combined_f1 = f1_score(all_bl_labels, all_bl_preds)

    # Side-by-side comparison
    common_users = set(baseline_f1s.keys()) & set(gnn_f1s.keys())
    baseline_common = [baseline_f1s[u] for u in common_users]
    gnn_common = [gnn_f1s[u] for u in common_users]

    print(f"=== Comparison (on {len(common_users)} common users) ===")
    print(f"  Baseline SVC mean F1: {np.mean(baseline_common):.4f}")
    print(f"  GNN mean F1:          {np.mean(gnn_common):.4f}")
    print(f"  GNN wins: {sum(g > b for g, b in zip(gnn_common, baseline_common))}/{len(common_users)}")
    print(f"\n  Baseline combined F1 (pooled): {combined_f1:.4f}")
    print(f"  GNN global F1 (pooled):        {global_f1:.4f}")

    # Side-by-side histogram
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

    axes[0].hist(baseline_common, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(np.mean(baseline_common), color='red', linestyle='--', label=f'Mean: {np.mean(baseline_common):.3f}')
    axes[0].set_xlabel('Test F1 Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Baseline SVC (RBF)')
    axes[0].legend()

    axes[1].hist(gnn_common, bins=20, edgecolor='black', alpha=0.7, color='darkorange')
    axes[1].axvline(np.mean(gnn_common), color='red', linestyle='--', label=f'Mean: {np.mean(gnn_common):.3f}')
    axes[1].set_xlabel('Test F1 Score')
    axes[1].set_title('GNN (General)')
    axes[1].legend()

    plt.suptitle(f'Per-user Test F1 Distribution: Baseline vs GNN — {FINAL_TAG}', fontsize=13)
    plt.tight_layout()
    plt.show()